# Tính năng 6 – Trang **Tư vấn & dự đoán lương** (trang chủ)

Yêu cầu của người dùng: *"Tôi muốn nhập kỹ năng và kinh nghiệm của mình; phần mềm dựa vào mô hình đã huấn luyện sẵn sẽ dự đoán mức lương và tư vấn kỹ năng nên học thêm."*

Người dùng nhập vai trò, khu vực, số năm kinh nghiệm, kỹ năng (và nếu muốn: chức danh, loại công ty, hình thức làm việc). Kết quả gồm:
- **khoảng lương** thường gặp và **mức điển hình** (nếu đủ dữ liệu), quy về lương gross thị trường gần đúng;
- tối đa **5 kỹ năng nên học**;
- **lộ trình lương** khi có thêm kinh nghiệm.

Mã nguồn: `src/career_market/model/advisor.py`, `model/support.py`, `pages/advisor.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Không huấn luyện lại khi người dùng bấm nút
Nếu huấn luyện lại mỗi lần bấm, người dùng phải chờ lâu và kết quả có thể mỗi lần một khác. Ứng dụng chỉ nạp mô hình đã lưu: cùng một hồ sơ thì hôm nay hay ngày mai đều ra cùng kết quả, và mô hình người dùng đang dùng chính là mô hình đã được kiểm định.

In [ ]:
import time
from career_market.model.advisor import advise, predict_profile
from career_market.model.estimators import MODEL_LABELS

backend = {"role": "Backend Developer", "region": "Hà Nội", "experience_years": 3, "skills": ["Java", "Spring Boot", "SQL"]}
start = time.perf_counter(); result = advise(backend); first = time.perf_counter() - start
start = time.perf_counter(); advise(backend); second = time.perf_counter() - start
print(f"Lần đầu (gồm nạp mô hình): {first:.2f} s · lần hai: {second:.2f} s")

## 2. Hồ sơ đủ dữ liệu: Backend · Hà Nội · 3 năm · Java, Spring Boot, SQL
Ứng dụng hiện **khoảng lương trước, con số sau**, giống dự báo thời tiết "26–32 độ" thay vì "29,3 độ". Khoảng ở đây chứa một nửa số tin tương tự (từ mức 25% đến mức 75%).

In [ ]:
def summary(result):
    m = result["market"]
    return pd.Series({
        "Cấp bậc suy ra": result["level"], "Đủ dữ liệu?": result["supported"], "Số tin cùng vai trò × cấp bậc": result["cohort_rows"],
        "Khoảng lương (gross gần đúng)": f"{result['low']:.1f} – {result['high']:.1f}",
        "Mức điển hình": "—" if result["salary"] is None else round(result["salary"], 1),
        "Trung vị thị trường của vai trò": m["role_median"], "Lý do (nếu ít dữ liệu)": result["support_message"] or "—",
    })

summary(result).to_frame("Backend 3 năm")

## 3. Cách chọn kỹ năng để gợi ý
1. Chỉ xét kỹ năng người dùng chưa có và xuất hiện ở ít nhất 3% số tin cùng vai trò (không gợi ý Figma cho lập trình viên Backend).
2. Thêm **đúng một** kỹ năng vào hồ sơ rồi dự đoán lại, để biết lương thay đổi bao nhiêu ("nếu tôi giữ nguyên mọi thứ nhưng biết thêm X").
3. Xếp theo mức tăng lương; nếu chưa đủ 5 kỹ năng thì bổ sung kỹ năng được tuyển nhiều nhất, có ghi rõ lý do.

Không xếp theo độ phổ biến, vì người dùng hỏi "học gì để tăng lương", không phải "cái gì đang phổ biến".

In [ ]:
recs = pd.DataFrame(result["recommendations"])
display(recs.rename(columns={"skill": "Kỹ năng", "uplift": "Tăng lương (triệu)", "demand_share": "Tỉ lệ tin yêu cầu", "reason": "Lý do"}))
top = result["recommendations"][0]["skill"]
manual = advise({**backend, "skills": backend["skills"] + [top]})["salary_range"]["median"] - result["salary_range"]["median"]
print(f"Tự kiểm tra {top}: bảng {result['recommendations'][0]['uplift']:+.2f} · tính tay {manual:+.2f}")
charts.recommendation_chart(result["recommendations"]);

## 4. Lộ trình lương theo kinh nghiệm

In [ ]:
display(pd.DataFrame(result["experience_path"]))
charts.experience_path_chart(result["experience_path"]);

Lương tăng chậm trong giai đoạn Middle rồi nhảy lên khi đến Senior (khoảng 5 năm), khớp với điều thấy ở trang Tổng quan. Quy tắc "không giảm" chỉ cấm lương đi xuống, không cấm đi ngang.

## 5. Hồ sơ ít dữ liệu: chỉ hiện khoảng lương

In [ ]:
data_engineer = {"role": "Data Engineer", "region": "Hà Nội", "experience_years": 3, "skills": ["Python", "SQL", "Spark"]}
low_data = advise(data_engineer)
display(summary(low_data).to_frame("Data Engineer 3 năm"))
pd.DataFrame(low_data["recommendations"])[["skill", "demand_share", "reason"]]

Một người tư vấn trung thực sẽ nói "với vị trí này tôi chỉ biết lương thường nằm trong khoảng X đến Y" thay vì bịa ra một con số. Với hồ sơ ít dữ liệu, trang cũng không hiện mức tăng lương của từng kỹ năng.

## 6. Bảo đảm "thêm kỹ năng không bao giờ làm giảm lương"

In [ ]:
from career_market.data.rules import SKILLS

profiles = [
    {"role": "Backend Developer", "region": "Hà Nội", "experience_years": 5, "skills": []},
    {"role": "Kiểm thử (QA/QC)", "region": "TP. Hồ Chí Minh", "experience_years": 1, "skills": ["SQL"]},
]
rows = []
for name in ["GradientBoosting", "Ridge", "RandomForest"]:
    worst = 0.0
    for p in profiles:
        base = predict_profile(p, name)["predicted"]
        for skill in SKILLS:
            if skill not in p["skills"]:
                worst = min(worst, predict_profile({**p, "skills": p["skills"] + [skill]}, name)["predicted"] - base)
    rows.append({"Mô hình": MODEL_LABELS[name], "Thay đổi nhỏ nhất khi thêm 1 kỹ năng (triệu)": round(worst, 4) + 0.0})
pd.DataFrame(rows)

Bộ kiểm thử tự động chạy đúng phép thử này trên mô hình đã lưu, để lỗi không quay lại sau mỗi lần huấn luyện lại.

## 7. Chia sẻ hồ sơ bằng đường dẫn
Đường dẫn `/?role=Backend Developer&region=Hà Nội&years=3&skills=Java,Spring Boot,SQL` mở thẳng kết quả. Giá trị không hợp lệ được bỏ qua thay vì làm trang báo lỗi.

In [ ]:
from urllib.parse import urlencode
from career_market.data.rules import REGIONS, ROLE_ORDER

def parse_query(q):
    shared = {"role": q.get("role") if q.get("role") in ROLE_ORDER else None,
              "region": q.get("region") if q.get("region") in REGIONS else None,
              "skills": [s for s in q.get("skills", "").split(",") if s in SKILLS]}
    try:
        shared["years"] = min(max(float(q.get("years", "")), 0.0), 20.0)
    except ValueError:
        shared["years"] = None
    return shared

print("http://localhost:8502/?" + urlencode({"role": "Backend Developer", "region": "Hà Nội", "years": 3, "skills": "Java,Spring Boot,SQL"}))
pd.DataFrame([parse_query({"role": "Backend Developer", "years": "3", "skills": "Java,Spring Boot,SQL"}),
              parse_query({"role": "Hacker", "years": "99", "skills": "Java,Excel97"})])

## Kết luận
- Trang tư vấn đáp ứng đủ yêu cầu của người dùng và nói thật về mức độ chắc chắn: khoảng trước, con số sau; ít dữ liệu thì chỉ có khoảng.
- Lời khuyên luôn hợp lý nhờ quy tắc "thêm kỹ năng không làm giảm lương".
- ⚠️ Mức tăng lương là quan hệ quan sát được, không phải lời hứa; lương là lương gross thị trường **gần đúng**, được hiệu chỉnh từ lương ghi trên tin tuyển dụng; mô hình chưa đạt chuẩn đưa vào sử dụng.